# LiteTorch Stage 4a Phase 1 — Real TPU Compute Spike (single submission, Save & Run All)

**What this proves (if it passes):** a standalone C++ program that talks to `libtpu` **directly** through the PJRT C API (no torch_xla, no JAX in the spike path) compiles a StableHLO matmul, executes it on the TPU chip, and matches the PyTorch reference.

**Roles — read carefully:**
- **Spike under test:** `spike_matmul` (PJRT C API + libtpu only).
- **Reference standard (numerics GATE):** PyTorch/XLA. The gate is spike vs PyTorch/XLA on identical 512x512 FP32 inputs: max abs diff < 1e-4.
- **JAX:** perf-ceiling reference (what a mature XLA stack tops out at on this chip) and a *secondary* numerics cross-check only. JAX is NOT the gate.
- **Host CPU (numpy):** the CPU perf signature the spike must be inconsistent with at large shapes.
- **Phase-6 shim reference:** 512^3 FP32 median 96.956 ms (labeled historical reference, not re-measured; the LiteTorch build is too expensive for this batch).

**Process isolation:** libtpu/PJRT init is process-global. The spike is its own binary; torch_xla and JAX each run as separate subprocesses writing partial JSONs. A baseline failure is recorded and never takes the spike results down.

**Phase 1 is single-chip only.** No multi-device / SPMD work (that is Phase 2).

## Cell 1 — Probe the Kaggle TPU image (libtpu, PJRT API version, packages, host)

In [ ]:
import ctypes, glob, importlib.metadata, json, os, platform, subprocess, sys

probe = {}
probe["python"] = sys.version
probe["platform"] = platform.platform()
probe["nproc"] = os.cpu_count()
try:
    with open("/proc/meminfo") as f:
        probe["meminfo_head"] = [next(f).strip() for _ in range(3)]
except Exception as e:
    probe["meminfo_error"] = str(e)
for pkg in ["torch", "torch_xla", "jax", "jaxlib", "libtpu", "numpy"]:
    try:
        probe["pkg_" + pkg] = importlib.metadata.version(pkg)
    except Exception:
        probe["pkg_" + pkg] = None
probe["env_PJRT_DEVICE"] = os.environ.get("PJRT_DEVICE")
probe["env_TPU_LIBRARY_PATH"] = os.environ.get("TPU_LIBRARY_PATH")
cands = []
for pat in ["/lib/libtpu.so", "/usr/lib/libtpu.so", "/usr/local/lib/libtpu.so",
            "/usr/local/lib/python3.*/dist-packages/libtpu/libtpu.so*",
            "/usr/local/lib/python3.*/site-packages/libtpu/libtpu.so*",
            os.path.expanduser("~/.local/lib/python3.*/site-packages/libtpu/libtpu.so*")]:
    cands.extend(glob.glob(pat))
probe["libtpu_candidates"] = [{"path": p, "size": os.path.getsize(p)} for p in sorted(set(cands))]
probe["pjrt_api_version_ctypes"] = None
if cands:
    try:
        lib = ctypes.CDLL(sorted(set(cands))[0])
        lib.GetPjrtApi.restype = ctypes.c_void_p
        api_ptr = lib.GetPjrtApi()
        major = ctypes.c_int.from_address(api_ptr + 32).value
        minor = ctypes.c_int.from_address(api_ptr + 36).value
        probe["pjrt_api_version_ctypes"] = str(major) + "." + str(minor)
    except Exception as e:
        probe["pjrt_api_version_ctypes_error"] = str(e)
print(json.dumps(probe, indent=2))
with open("/kaggle/working/probe.json", "w") as f:
    json.dump(probe, f, indent=2)

## Cell 2 — Fetch run inputs from kaggle-run-inputs (GitHub-fetch only), sha256-verify every file, build the spike

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, urllib.request

MANIFEST = {
  "spike_matmul.cpp": "b589b120320bfea7c2e5ca6f2bd7eb33bae6d2c07e81da28cc88f6a49ba6e610",
  "pjrt_c_api.h": "8c54c8e030ea88fcba627dd83286e09edbc65aaa34f7aced36da51a48fe83328",
  "run_spike.py": "0804912de17603fca4a7c6b1266fb831c07f813e2d030d567d285fbb8e1a8830",
  "bench_torch_xla.py": "746d370dd1a7df4647d999a15e9d150a104ecd53ad3a725119d327e4b0f6e249",
  "bench_jax.py": "ae960f956be94242e243a7edd0d0045a9e708d7ff569412fada2d116840ecb9d",
  "bench_cpu.py": "ce2dd63304585e7399517f67621c3f7b7bbbf11ceddc252b7f4ea25b3d69e368"
}
REPO = "https://github.com/nguyenminh20000/Litetorch-"
RAW = "https://raw.githubusercontent.com/nguyenminh20000/Litetorch-/kaggle-run-inputs/tpu-spike/"
SRC = "/kaggle/working/tpu-spike-src"
os.makedirs(SRC, exist_ok=True)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

fetched = {}
clone_dir = "/kaggle/working/litetorch-fetch"
if not os.path.exists(os.path.join(clone_dir, "tpu-spike")):
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", "kaggle-run-inputs", REPO, clone_dir],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
for name in MANIFEST:
    dst = os.path.join(SRC, name)
    src_clone = os.path.join(clone_dir, "tpu-spike", name)
    if os.path.exists(src_clone):
        shutil.copyfile(src_clone, dst)
    else:
        urllib.request.urlretrieve(RAW + name, dst)
    got = sha256_of(dst)
    assert got == MANIFEST[name], "SHA256 MISMATCH for " + name + ": " + got
    fetched[name] = got
    print("verified", name, got[:12])
build = subprocess.run(["g++", "-std=c++14", "-O2", "-o", "/kaggle/working/spike_matmul",
                        os.path.join(SRC, "spike_matmul.cpp"), "-ldl"],
                       capture_output=True, text=True)
print(build.stdout, build.stderr)
assert build.returncode == 0, "spike build failed"
print("BUILD OK -> /kaggle/working/spike_matmul")

## Cell 3 — Run the spike (own process): FP32 gate run on shared inputs, FP32 512/4096 timing, BF16 4096 + rect timing, BF16 sanity

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/tpu-spike-src/run_spike.py",
                    "--bin", "/kaggle/working/spike_matmul", "--workdir", "/kaggle/working"],
                   capture_output=True, text=True)
print(r.stdout[-8000:])
print(r.stderr[-3000:])
print("run_spike returncode:", r.returncode)

## Cell 4 — PyTorch/XLA baseline (separate subprocess): reference standard + gate counterpart on the shared inputs

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/tpu-spike-src/bench_torch_xla.py",
                    "--workdir", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("bench_torch_xla returncode:", r.returncode)

## Cell 5 — JAX baseline (separate subprocess): perf ceiling + secondary cross-check only

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/tpu-spike-src/bench_jax.py",
                    "--workdir", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("bench_jax returncode:", r.returncode)

## Cell 6 — Host CPU signature (separate subprocess, numpy FP32)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/tpu-spike-src/bench_cpu.py",
                    "--workdir", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("bench_cpu returncode:", r.returncode)

## Cell 7 — Merge partials, evaluate the numerics gate and the perf-signature criterion, write results.json

In [ ]:
import json, os
import numpy as np

W = "/kaggle/working"
def load(name):
    p = os.path.join(W, name)
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return {"error": "partial file missing: " + name}

results = {}
results["stage"] = "4a-phase1-spike"
results["roles"] = {
    "spike_under_test": "spike_matmul via PJRT C API against libtpu directly; no torch_xla, no JAX in this path",
    "numerics_gate": "PyTorch/XLA is the reference standard; gate = spike vs PyTorch/XLA on identical 512x512 FP32 inputs, max abs diff < 1e-4",
    "jax": "perf ceiling reference and secondary numerics cross-check only; NOT the gate",
    "host_cpu": "numpy FP32 signature for the perf-inconsistency criterion",
    "shim_reference": {"label": "Phase-6 shim path, historical reference, not re-measured in this run",
                       "f32_512_median_ms": 96.956},
}
results["probe"] = load("probe.json")
results["spike"] = load("partial_spike.json")
results["torch_xla"] = load("partial_torch_xla.json")
results["jax"] = load("partial_jax.json")
results["cpu"] = load("partial_cpu.json")

gate = {"status": "not_evaluated"}
spike_out = os.path.join(W, "out_spike_512.bin")
torch_out = os.path.join(W, "out_torch_xla_512.bin")
if os.path.exists(spike_out) and os.path.exists(torch_out):
    s = np.fromfile(spike_out, dtype=np.float32)
    t = np.fromfile(torch_out, dtype=np.float32)
    if s.shape == t.shape and s.size == 512 * 512:
        d = float(np.abs(s - t).max())
        gate = {"spike_vs_torch_xla_512_max_abs_diff": d, "tolerance": 1e-4,
                "status": "PASS" if d < 1e-4 else "FAIL"}
results["numerics_gate_spike_vs_torch_xla"] = gate

crit = {"status": "not_evaluated"}
try:
    spike_4096 = results["spike"]["runs"]["f32_4096"]["result"]["execute_median_ms"]
    cpu_4096 = results["cpu"]["runs"]["f32_4096"]["execute_median_ms"]
    crit = {"spike_f32_4096_median_ms": spike_4096, "cpu_f32_4096_median_ms": cpu_4096,
            "ratio_cpu_over_spike": cpu_4096 / spike_4096,
            "status": "PASS" if spike_4096 < 0.25 * cpu_4096 else "CHECK"}
except Exception as e:
    crit = {"status": "not_evaluated", "reason": str(e)}
results["perf_signature_criterion"] = crit

with open(os.path.join(W, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps({"numerics_gate": gate, "perf_signature": crit}, indent=2))
print("results.json written to /kaggle/working/results.json")